SINGLE-TASK BASELINE EXPERIMENT

In [1]:
# Root
import sys
import os

sys.path.append(
    os.path.abspath("..")
)

Import

In [21]:
import torch
import random

from src.models.single_task_model import (
    SingleTaskModel
)

from src.datasets.single_task_dataset import (
    SingleTaskDataset
)

from src.datasets.transforms import (
    get_train_transforms
)

from torch.utils.data import (
    DataLoader,
    Subset
)

from src.losses.masked_loss import (
    MaskedBCELoss
)

from src.training.train import (
    train_one_epoch
)

from src.training.validate import (
    validate_one_epoch
)

from src.evaluation.metrics import (
    calculate_metrics
)

Device

In [3]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(device)

cpu


Load Model

In [4]:
model = SingleTaskModel()

model = model.to(device)

print(model)

SingleTaskModel(
  (backbone): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1

Dummy Input

In [5]:
dummy_input = torch.randn(
    16,
    3,
    224,
    224
).to(device)

Forward Pass

In [6]:
outputs = model(dummy_input)

print(outputs.shape)

torch.Size([16, 1])


Load Dataset TB

In [7]:
dataset = SingleTaskDataset(
    csv_file="../data/train/train.csv",
    target_label="tb",
    transform=get_train_transforms()
)

print(len(dataset))

240681


Ambil Sample

In [8]:
image, label, mask = dataset[0]

print(image.shape)

print(label)

print(mask)

torch.Size([3, 224, 224])
tensor([0.])
tensor([0.])


TRAIN SINGLE-TASK TB MODEL

In [9]:
# Buat Small Subset
small_dataset = Subset(
    dataset,
    range(100)
)

DataLoader

In [10]:
train_loader = DataLoader(
    small_dataset,
    batch_size=16,
    shuffle=True
)

Criterion

In [11]:
criterion = MaskedBCELoss()

Optimizer

In [12]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

Jalankan 1 Epoch

In [13]:
train_loss = train_one_epoch(
    model=model,
    dataloader=train_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device
)

print(train_loss)

0.16384300163814


Tambahkan Validation Dataset

In [14]:
val_dataset = SingleTaskDataset(
    csv_file="../data/val/val.csv",
    target_label="tb",
    transform=get_train_transforms()
)

Small Validation Subset

In [28]:
random_indices = random.sample(
    range(len(val_dataset)),
    1000
)

small_val_dataset = Subset(
    val_dataset,
    random_indices
)

Validation Loader

In [29]:
val_loader = DataLoader(
    small_val_dataset,
    batch_size=16,
    shuffle=False
)

Jalankan Validation

In [30]:
val_loss = validate_one_epoch(
    model=model,
    dataloader=val_loader,
    criterion=criterion,
    device=device
)

print(val_loss)

0.6157833867602878


Evaluasi Predictions

In [31]:
model.eval()

y_true = []

y_pred = []

y_prob = []

with torch.no_grad():

    for images, labels, mask in val_loader:

        images = images.to(device)

        labels = labels.to(device)

        outputs = model(images)

        probs = torch.sigmoid(outputs)

        preds = (probs > 0.5).float()

        y_true.extend(
            labels.cpu().numpy().flatten()
        )

        y_pred.extend(
            preds.cpu().numpy().flatten()
        )

        y_prob.extend(
            probs.cpu().numpy().flatten()
        )

Hitung Metrics

In [32]:
results = calculate_metrics(
    y_true,
    y_pred,
    y_prob
)

print(results)

{'accuracy': 0.065, 'precision': 0.0, 'recall': 0.0, 'f1_score': 0.0, 'roc_auc': 0.06406406406406406, 'specificity': np.float64(0.06506506506506507)}


CEK DISTRIBUSI LABEL

In [33]:
all_labels = []

for _, label, mask in small_val_dataset:

    if mask.item() == 1:

        all_labels.append(
            label.item()
        )

print(set(all_labels))

print(
    "Total valid labels:",
    len(all_labels)
)

print(
    "Positive:",
    sum(all_labels)
)

print(
    "Negative:",
    len(all_labels) - sum(all_labels)
)

{0.0, 1.0}
Total valid labels: 22
Positive: 1.0
Negative: 21.0
